# Camada Silver: canais, motivos e atendentes



In [0]:
from pyspark.sql import functions as F, Window

catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

## Funções auxiliares

In [0]:
ACENTOS = "áàâãäéèêëíìîïóòôõöúùûüç"
SEM_ACENTO = "aaaaaeeeeiiiiooooouuuuc"


def chave(c):
    """Texto em minúsculas, sem acento, sem espaços nas pontas e com espaços internos únicos.
    Serve só para comparar grafias; o valor final vem do dicionário de mapeamento."""
    t = F.lower(F.trim(F.col(c).cast("string")))
    return F.regexp_replace(F.translate(t, ACENTOS, SEM_ACENTO), r"\s+", " ")


def mapear(c, mapa):
    """Troca o valor da coluna `c` pelo valor do dicionário. Valor fora do dicionário vira null
    (e as validações abaixo barram o notebook se isso acontecer)."""
    expr = None
    for k, v in mapa.items():
        expr = F.when(F.col(c) == k, v) if expr is None else expr.when(F.col(c) == k, v)
    return expr


def ultima_versao(df, chaves):
    """Fica com a carga mais recente de cada chave (a Bronze é append) e descarta ingestion_datetime."""
    w = Window.partitionBy(*chaves).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "ingestion_datetime")


def data_iso(c):
    """Converte texto dd/MM/yyyy em date. Vazio ou fora do padrão vira null."""
    t = F.trim(F.col(c).cast("string"))
    return F.when(t.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(t, "dd/MM/yyyy"))


def checar(regra, df_falhas):
    """Para o notebook se alguma linha violar a regra."""
    n = df_falhas.count()
    assert n == 0, f"FALHA na validação '{regra}': {n} linha(s)"
    print(f"OK: {regra}")


def salvar(df, nome):
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.{nome}"))
    print(f"{silver_schema}.{nome}: {spark.table(f'{silver_schema}.{nome}').count()} linhas")

## 1. Canais

In [0]:
# Inclui variações vistas em chamados.canal, para reaproveitar o mesmo dicionário na Silver de chamados
MAPA_CANAL = {
    "ura": "URA", "u.r.a": "URA",
    "chatbot": "Chatbot", "chat bot": "Chatbot",
    "atendimento inicial": "Atendimento Inicial", "atendimento_inicial": "Atendimento Inicial",
    "atend. inicial": "Atendimento Inicial",
    "atendimento especializado": "Atendimento Especializado", "atendimento_especializado": "Atendimento Especializado",
    "atend. especializado": "Atendimento Especializado",
}
MAPA_TIPO_CANAL = {"autosservico": "Autosserviço", "humano": "Humano"}

b_canais = spark.table(f"{bronze_schema}.canais")
display(b_canais)

In [0]:
canais = ultima_versao(
    b_canais.withColumn("id_canal", F.trim(F.col("id_canal")).cast("int")),
    ["id_canal"],
)
canais = (
    canais
    .withColumn("k_nome", chave("nome_canal"))
    .withColumn("k_tipo", chave("tipo_canal"))
    .withColumn("nome_canal", mapear("k_nome", MAPA_CANAL))
    .withColumn("tipo_canal", mapear("k_tipo", MAPA_TIPO_CANAL))
    .withColumn("status_canal", F.lower(F.trim(F.col("status_canal"))))
    .select("id_canal", "nome_canal", "tipo_canal", "status_canal")
    .orderBy("id_canal")
)

checar("canais: nenhum campo nulo", canais.filter(
    F.col("id_canal").isNull() | F.col("nome_canal").isNull()
    | F.col("tipo_canal").isNull() | F.col("status_canal").isNull()))
checar("canais: id_canal único", canais.groupBy("id_canal").count().filter("count > 1"))
checar("canais: status é ativo ou inativo", canais.filter(~F.col("status_canal").isin("ativo", "inativo")))
display(canais)

## 2. Motivos

In [0]:
MAPA_CATEGORIA = {
    "cartao": "Cartão", "beneficios": "Benefícios", "financeiro": "Financeiro",
    "cadastral": "Cadastral", "atendimento": "Atendimento",
}
MAPA_CRITICIDADE = {"alta": "Alta", "media": "Média", "baixa": "Baixa"}

# Só para os motivos sem categoria na origem (chave = nome do motivo sem acento, em minúsculas)
IMPUTACAO_CATEGORIA = {
    "transferencia de agencia": "Cadastral",
    "contratacao de cartao adicional": "Cartão",
    "cancelamento de conta": "Financeiro",
}

b_motivos = spark.table(f"{bronze_schema}.base_motivos")
display(b_motivos)

In [0]:
motivos = ultima_versao(
    b_motivos.withColumn("id_motivo", F.trim(F.col("id_motivo")).cast("int")),
    ["id_motivo"],
)
motivos = (
    motivos
    .withColumn("nome_motivo", F.trim(F.col("nome_motivo")))
    .withColumn("k_nome", chave("nome_motivo"))
    .withColumn("k_cat", chave("categoria"))
    .withColumn("k_crit", chave("criticidade"))
    .withColumn("categoria_imputada", F.col("k_cat").isNull() | (F.col("k_cat") == ""))         # sem categoria na origem
    .withColumn("categoria", F.coalesce(
        mapear("k_cat", MAPA_CATEGORIA),
        F.when(F.col("categoria_imputada"), mapear("k_nome", IMPUTACAO_CATEGORIA)),
    ))
    .withColumn("criticidade", mapear("k_crit", MAPA_CRITICIDADE))
    .select("id_motivo", "nome_motivo", "categoria", "criticidade", "categoria_imputada")
    .orderBy("id_motivo")
)

checar("motivos: nenhum campo nulo", motivos.filter(
    F.col("id_motivo").isNull() | F.col("nome_motivo").isNull()
    | F.col("categoria").isNull() | F.col("criticidade").isNull()))
checar("motivos: id_motivo único", motivos.groupBy("id_motivo").count().filter("count > 1"))
checar("motivos: categoria é uma das 5 do case", motivos.filter(
    ~F.col("categoria").isin("Financeiro", "Cartão", "Cadastral", "Atendimento", "Benefícios")))
display(motivos)

## 3. Atendentes

In [0]:
b_atendentes = spark.table(f"{bronze_schema}.base_atendentes")
display(b_atendentes)

In [0]:
atendentes = ultima_versao(
    b_atendentes.withColumn("id_atendente", F.trim(F.col("id_atendente")).cast("int")),
    ["id_atendente"],
)
atendentes = (
    atendentes
    .withColumn("nome_atendente", F.trim(F.col("nome_atendente")))
    # pega o dígito: '1', 'Nível 1', '2', 'Nível 2', 'N2' -> 1 ou 2
    .withColumn("nivel_atendimento", F.when(
        F.col("nivel_atendimento").rlike(r"\d"),
        F.regexp_extract(F.col("nivel_atendimento"), r"(\d)", 1).cast("int")))
    .withColumn("data_admissao", data_iso("data_admissao"))
    .withColumn("data_desligamento", data_iso("data_desligamento"))
    .withColumn("status_atendente", F.lower(F.trim(F.col("status_atendente"))))
    .select("id_atendente", "nome_atendente", "nivel_atendimento",
            "data_admissao", "data_desligamento", "status_atendente")
    .orderBy("id_atendente")
)

checar("atendentes: id, nome, nível, admissão e status preenchidos", atendentes.filter(
    F.col("id_atendente").isNull() | F.col("nome_atendente").isNull()
    | F.col("nivel_atendimento").isNull() | F.col("data_admissao").isNull()
    | F.col("status_atendente").isNull()))
checar("atendentes: id_atendente único", atendentes.groupBy("id_atendente").count().filter("count > 1"))
checar("atendentes: nível é 1 ou 2", atendentes.filter(~F.col("nivel_atendimento").isin(1, 2)))
checar("atendentes: desligamento não é anterior à admissão",
       atendentes.filter(F.col("data_desligamento") < F.col("data_admissao")))
checar("atendentes: inativo se e somente se tem data de desligamento", atendentes.filter(
    (F.col("status_atendente") == "inativo") != F.col("data_desligamento").isNotNull()))
display(atendentes)

## 4. Gravação da Silver
Só chega aqui se todas as validações passaram.

In [0]:
salvar(canais, "tb_canais")
salvar(motivos, "tb_base_motivos")
salvar(atendentes, "tb_base_atendentes")